# TTM H-DATA-S1 — NEW FINAL24 owner one-shot release (2026-11-14)

**Owner GO recorded separately:** `GO_HDATA_S1_NEW_FINAL24_ONE_SHOT_OWNER_INFERENCE`.
This is an owner-executed Colab notebook, not an assistant-triggered run. Only Tesla T4 with frozen Torch 2.14.1+cu130 is admitted.
The fixed candidate SHA-256 and permit SHA-256 are verified before access. Set `HDATA_FINAL24_RUN_NOW=True` and run Cell 2 **once** only. The candidate rechecks all 24 secret prompt hashes, context and BPE roundtrip before GPU model-forward.
Do not run if another science job uses T4. Do not rerun this cell or delete a partially created output root after an error. Do not share the AES key, mapping or private owner prompt file. Do not score/unblind yet.


In [ ]:
# TTM H-DATA-S1 NEW FINAL24 — EXPLICIT OWNER MANUAL ONE-SHOT
# The owner already authorized this *separate* FINAL24 generation.
# For fail-closed safety, set this ONE flag to True and run once.
# Do NOT rerun after a failed child or an existing generation root.
HDATA_FINAL24_RUN_NOW = False

import hashlib, pathlib, subprocess, sys
from collections import deque

def hdata_final_need(ok, why):
    if not ok:
        raise RuntimeError("HDATA_S1_FINAL24_OWNER_HOLD:" + why)

def hdata_final_sha(path):
    h = hashlib.sha256()
    with pathlib.Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1048576), b""):
            h.update(chunk)
    return h.hexdigest()

if not HDATA_FINAL24_RUN_NOW:
    print("HDATA_S1_FINAL24_OWNER_DISPATCH=NOT_REQUESTED")
    print("Owner GO is recorded; set HDATA_FINAL24_RUN_NOW=True in this cell only.")
    print("HDATA_S1_FINAL24_INFERENCE=NOT_EXECUTED")
    print("HDATA_S1_UNBLIND=NOT_EXECUTED")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    base = pathlib.Path("/content/drive/MyDrive/TTM/ttm-small-pilot")
    owner = base / "ttm_small4m_h_data_s1_20261113"
    private = owner / "owner_private"
    science = owner / "science_one_shot_v1"
    final_root = owner / "new_final24_blind_generation_once_v1"

    # Fail without touching FINAL plaintext or AES if an old run exists.
    hdata_final_need(science.is_dir(), "SCIENCE_ROOT_MISSING")
    hdata_final_need(private.is_dir(), "OWNER_PRIVATE_ROOT_MISSING")
    hdata_final_need(not final_root.exists() and not final_root.is_symlink(),
                     "FINAL_ONE_SHOT_ALREADY_EXISTS_NO_REPLAY")
    hdata_final_need((science / "ARM_H0" / "checkpoint_1152" / "state.pt").is_file(),
                     "H0_FINAL_CHECKPOINT_MISSING")
    hdata_final_need((science / "ARM_H1" / "checkpoint_1152" / "state.pt").is_file(),
                     "H1_FINAL_CHECKPOINT_MISSING")

    release_repo = pathlib.Path("/content/TTM_HDATA_S1_FINAL24_RELEASE_PINNED")
    frozen_head = "9ed7116b53eeb77ddbcc4b5728c81a6a971f8dfe"
    if not release_repo.exists():
        subprocess.run([
            "git", "clone", "--quiet",
            "https://github.com/minhtri22/TokenToModel-Lab.git",
            str(release_repo)
        ], check=True)
        subprocess.run(["git", "checkout", "--detach", frozen_head],
                       cwd=release_repo, check=True)
    hdata_final_need((release_repo / ".git").is_dir(),
                     "INVALID_RELEASE_REPO")
    head = subprocess.check_output(["git", "rev-parse", "HEAD"],
                                   cwd=release_repo, text=True).strip()
    hdata_final_need(head == frozen_head, "RELEASE_CHECKOUT_SHA_DRIFT")

    source = release_repo / (
        "tools/pilot_t4/hdata_s1/"
        "ttm_hdata_s1_new_final24_blind_generation_candidate_source_only_v1.py"
    )
    permit = release_repo / (
        "artifacts/qa/ttm-small4m-h-data-s1/science_postrun/"
        "TTM_HDATA_S1_OWNER_NEW_FINAL24_ONE_SHOT_INFERENCE_GO_PERMIT_20261114.json"
    )
    source_sha = "7953d33fa592a8890a185cedf490d7af2df9514e025395f00a1996c7bf784363"
    permit_sha = "242a0925320e9c4aeb716fca4324cdce12d384f52322589f2e403ae464702581"
    hdata_final_need(hdata_final_sha(source) == source_sha,
                     "PINNED_GENERATOR_SHA_DRIFT")
    hdata_final_need(hdata_final_sha(permit) == permit_sha,
                     "OWNER_PERMIT_SHA_DRIFT")

    # No training; independently verify the public permit/source gate.
    result = subprocess.run([
        sys.executable, str(source), "--mode", "source-only"
    ], cwd=release_repo, check=True, capture_output=True, text=True)
    hdata_final_need("SOURCE_ONLY_FINAL_OWNER_RELEASE_NOT_GRANTED" in result.stdout,
                     "SOURCE_ONLY_DEFAULT_GATE_DRIFT")

    def find_private(name, expected):
        # Search only inside the previously locked pilot Drive subtree.
        candidates = sorted(p for p in base.rglob(name) if
                            p.is_file() and not p.is_symlink())
        hits = [p for p in candidates if hdata_final_sha(p) == expected]
        hdata_final_need(bool(hits), "MISSING_PINNED_FILE:" + name)
        return hits[0]

    bpe = find_private(
        "T1_BPE2048_REAL_TRAIN_MANIFEST.json",
        "cba18e60cee3d08a353ee6d4a451017289e45af7b4b1179c87171b9b78d7dfdc"
    )
    parent = find_private(
        "checkpoint.pt",
        "cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b"
    )
    owner_final = private / "TTM_HDATA_S1_OWNER_PRIVATE_DEV12_FINAL24_20261113.json"
    owner_escrow = private / "TTM_HDATA_S1_OWNER_PRIVATE_ESCROW_BACKUP_DO_NOT_GITHUB.zip"
    hdata_final_need(owner_final.is_file() and owner_escrow.is_file(),
                     "FROZEN_PRIVATE_FINAL_OR_ESCROW_MISSING")

    h0 = science / "ARM_H0" / "checkpoint_1152" / "state.pt"
    h1 = science / "ARM_H1" / "checkpoint_1152" / "state.pt"
    hdata_final_need(hdata_final_sha(h0) ==
                     "a7cdade3b70cb2ceed258edd1e3edd494ce9745633d2e528ee90232f3c95c2fe",
                     "H0_CHECKPOINT_SHA")
    hdata_final_need(hdata_final_sha(h1) ==
                     "31d9365a470e638f77e6661252ed8d0605e9f9095da145d9f48751adcdcfcb64",
                     "H1_CHECKPOINT_SHA")
    hdata_final_need(not final_root.exists(),
                     "FINAL_ROOT_APPEARED_NO_DISPATCH")

    args = [
        sys.executable, str(source),
        "--mode", "generate",
        "--permit", str(permit),
        "--permit-sha", permit_sha,
        "--frozen-repo", str(release_repo),
        "--parent", str(parent),
        "--bpe", str(bpe),
        "--owner-final", str(owner_final),
        "--owner-escrow-zip", str(owner_escrow),
        "--h0-checkpoint", str(h0),
        "--h1-checkpoint", str(h1)
    ]
    print("HDATA_S1_FINAL24_EXACT_OWNER_RELEASE=PASS", flush=True)
    print("HDATA_S1_FINAL24_GENERATION=EXECUTING_ONE_SHOT", flush=True)
    tail = deque(maxlen=70)
    with subprocess.Popen(args, cwd=release_repo,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as child:
        for line in child.stdout:
            tail.append(line)
            print(line, end="", flush=True)
        ret = child.wait()
    if ret:
        print("HDATA_S1_FINAL24_EXIT_CODE=" + str(ret))
        print("HDATA_S1_CHILD_LAST_70_LINES_BEGIN")
        print("".join(tail))
        print("HDATA_S1_CHILD_LAST_70_LINES_END")
        print("HDATA_S1_FINAL24=HOLD_KEEP_ALL_PARTIAL_EVIDENCE_NO_RERUN")
        raise RuntimeError("H_DATA_S1_FINAL24_ONE_SHOT_FAILED_DO_NOT_REPLAY")

    packet = final_root / "BLIND_FINAL24_RATER_PACKET.json"
    receipt = final_root / "OWNER_PRIVATE_GENERATION_RECEIPT.json"
    hdata_final_need(packet.is_file() and receipt.is_file(),
                     "OWNER_FINAL_PACKET_OR_RECEIPT_MISSING")
    print("HDATA_S1_FINAL24_PACKET_SHA256=" + hdata_final_sha(packet))
    print("HDATA_S1_FINAL24_RECEIPT_SHA256=" + hdata_final_sha(receipt))
    print("HDATA_S1_BLIND_SCORING=NOT_EXECUTED")
    print("HDATA_S1_UNBLIND=NOT_EXECUTED")
    print("HDATA_S1_KEEP_PRIVATE_PACKET_ON_DRIVE=TRUE")
